# Batch Paper Question Answering

This notebook uses `paper_qa_with_attributions.ipynb` as the retrieval/answering implementation, then runs it over `paper_questions.json`.

Output rows are appended to a list and serialized as JSON. Each row has the requested task shape plus batch metadata:

`paper_title`, `paper_id`, `question`, `answer`, `attribute_uris`, and mapping status fields.

## Configuration

Set `MAX_ITEMS` to a small integer for a quick dry run, or leave it as `None` to process every paper-title/question pair.

In [3]:
from __future__ import annotations

import json
import logging
import re
import sys
from difflib import SequenceMatcher
from pathlib import Path
from typing import Any

logging.getLogger("uri_resolver").setLevel(logging.WARNING)
logging.getLogger("uri_resolver.doc").setLevel(logging.WARNING)
logging.getLogger("uri_resolver.fuseki").setLevel(logging.WARNING)

def find_repo_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (
            (candidate / "paper_questions.json").exists()
            and (candidate / "notebooks" / "paper_qa_with_attributions.ipynb").exists()
        ):
            return candidate
    raise FileNotFoundError(
        "Could not find repo root containing paper_questions.json and "
        "notebooks/paper_qa_with_attributions.ipynb"
    )


ROOT = find_repo_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
QUESTIONS_PATH = ROOT / "paper_questions.json"
PREVIOUS_NOTEBOOK_PATH = ROOT / "notebooks" / "paper_qa_with_attributions.ipynb"
OUTPUT_PATH = ROOT / "notebooks" / "paper_qa_batch_results.json"

MAX_ITEMS: int | None = None
TOP_K = 12
ANSWER_CHUNKS = 4
ANSWER_MODE = "openrouter"  # Use "extractive" for the previous non-LLM behavior.
FUZZY_MATCH_THRESHOLD = 0.88

# Use this when a question title differs from the KG title. Values may be local paper ids or full paper URIs.
MANUAL_TITLE_TO_PAPER_ID: dict[str, str] = {}


## Load The Previous Notebook Functions

This executes the setup and helper cells from `paper_qa_with_attributions.ipynb`, while skipping its example cells.

In [4]:
def load_previous_notebook_functions(path: Path) -> None:
    notebook = json.loads(path.read_text())
    namespace = globals()
    skipped_markers = (
        "PAPER_ID =",
        "QUESTION =",
        "result = answer_question",
        "retrieve_relevant_chunks(PAPER_ID",
    )
    for cell in notebook["cells"]:
        if cell.get("cell_type") != "code":
            continue
        source = "".join(cell.get("source", []))
        if any(marker in source for marker in skipped_markers):
            continue
        exec(compile(source, str(path), "exec"), namespace)


load_previous_notebook_functions(PREVIOUS_NOTEBOOK_PATH)

# Smoke check that the previous notebook's public functions are available.
[name for name in ["answer_question", "run_select", "paper_uri_from_id"] if name in globals()]


['answer_question', 'run_select', 'paper_uri_from_id']

## Read And Flatten `paper_questions.json`

In [5]:
def read_question_pairs(path: Path) -> list[dict[str, str]]:
    payload = json.loads(path.read_text())
    entries = payload.get("data", []) if isinstance(payload, dict) else payload
    if not isinstance(entries, list):
        raise ValueError("paper_questions.json must contain a list or a {'data': [...]} object")

    pairs: list[dict[str, str]] = []
    for entry in entries:
        if not isinstance(entry, dict):
            continue
        paper_title = str(entry.get("paper_title", "")).strip()
        questions = entry.get("questions", [])
        if not paper_title or not isinstance(questions, list):
            continue
        for question in questions:
            question_text = str(question).strip()
            if question_text:
                pairs.append({"paper_title": paper_title, "question": question_text})
    return pairs


question_pairs = read_question_pairs(QUESTIONS_PATH)
question_pairs = question_pairs[:MAX_ITEMS] if MAX_ITEMS is not None else question_pairs

{
    "paper_entries": len(json.loads(QUESTIONS_PATH.read_text()).get("data", [])),
    "question_pairs": len(question_pairs),
    "first_pair": question_pairs[0] if question_pairs else None,
}


{'paper_entries': 12,
 'question_pairs': 120,
 'first_pair': {'paper_title': 'Towards Faithfully Interpretable NLP Systems: How should we define and evaluate faithfulness?',
  'question': 'What exactly do the authors mean by "faithfulness" in NLP interpretability?'}}

## Build Paper Title To Paper ID Mapping

This cell queries Fuseki for paper titles, then builds exact and fuzzy lookup tables. It also supports `MANUAL_TITLE_TO_PAPER_ID` for titles whose KG title differs from `paper_questions.json`.

In [6]:
def normalize_title(title: str) -> str:
    title = title.casefold().strip()
    title = re.sub(r"[^a-z0-9]+", " ", title)
    return " ".join(title.split())


def local_paper_id_from_uri(uri: str) -> str:
    marker = "/paper/"
    return uri.rsplit(marker, 1)[1] if marker in uri else uri.rstrip("/").rsplit("/", 1)[-1]


def fetch_kg_papers() -> list[dict[str, str]]:
    query = """
PREFIX dct: <http://purl.org/dc/terms/>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>

SELECT DISTINCT ?paper ?title WHERE {
  GRAPH ?g {
    ?paper dct:title ?title .
    FILTER(CONTAINS(STR(?paper), "/paper/"))
  }
}
ORDER BY LCASE(STR(?title))
"""
    rows = run_select(query)
    papers: list[dict[str, str]] = []
    seen: set[tuple[str, str]] = set()
    for row in rows:
        paper_uri = binding_value(row, "paper")
        title = binding_value(row, "title")
        if not paper_uri or not title:
            continue
        key = (paper_uri, title)
        if key in seen:
            continue
        seen.add(key)
        papers.append(
            {
                "paper_uri": paper_uri,
                "paper_id": local_paper_id_from_uri(paper_uri),
                "title": title,
                "normalized_title": normalize_title(title),
            }
        )
    return papers


kg_papers = fetch_kg_papers()
exact_title_map = {paper["normalized_title"]: paper for paper in kg_papers}

{
    "kg_papers": len(kg_papers),
    "sample": kg_papers[:3],
}


{'kg_papers': 276,
 'sample': [{'paper_uri': 'https://w3id.org/twc/sudo/kg/paper/what_is_relevant_in_a_text_document_an_interpretable_machine_learning_approach',
   'paper_id': 'what_is_relevant_in_a_text_document_an_interpretable_machine_learning_approach',
   'title': '"What is relevant in a text document?": An interpretable machine learning approach',
   'normalized_title': 'what is relevant in a text document an interpretable machine learning approach'},
  {'paper_uri': 'https://w3id.org/twc/sudo/kg/paper/will_you_find_these_shortcuts_a_protocol_for_evaluating_the_faithfulness_of_input_salience_methods_for_text_classification',
   'paper_id': 'will_you_find_these_shortcuts_a_protocol_for_evaluating_the_faithfulness_of_input_salience_methods_for_text_classification',
   'title': '"Will You Find These Shortcuts?" A Protocol for Evaluating the Faithfulness of Input Salience Methods for Text Classification',
   'normalized_title': 'will you find these shortcuts a protocol for evaluatin

In [7]:
def manual_mapping_record(title: str, mapped_value: str) -> dict[str, Any]:
    paper_uri = paper_uri_from_id(mapped_value)
    return {
        "paper_uri": paper_uri,
        "paper_id": local_paper_id_from_uri(paper_uri),
        "title": title,
        "normalized_title": normalize_title(title),
        "mapping_method": "manual",
        "mapping_score": 1.0,
    }


def best_fuzzy_title_match(normalized_title: str) -> tuple[dict[str, str] | None, float]:
    best_paper: dict[str, str] | None = None
    best_score = 0.0
    for paper in kg_papers:
        score = SequenceMatcher(None, normalized_title, paper["normalized_title"]).ratio()
        if score > best_score:
            best_score = score
            best_paper = paper
    return best_paper, best_score


def map_title_to_paper(title: str) -> dict[str, Any]:
    normalized = normalize_title(title)
    manual_value = MANUAL_TITLE_TO_PAPER_ID.get(title) or MANUAL_TITLE_TO_PAPER_ID.get(normalized)
    if manual_value:
        return manual_mapping_record(title, manual_value)

    exact = exact_title_map.get(normalized)
    if exact:
        return {**exact, "mapping_method": "exact", "mapping_score": 1.0}

    fuzzy, score = best_fuzzy_title_match(normalized)
    if fuzzy and score >= FUZZY_MATCH_THRESHOLD:
        return {**fuzzy, "mapping_method": "fuzzy", "mapping_score": score}

    return {
        "paper_uri": None,
        "paper_id": None,
        "title": None,
        "normalized_title": normalized,
        "mapping_method": "unmatched",
        "mapping_score": score,
        "best_candidate_title": fuzzy["title"] if fuzzy else None,
        "best_candidate_paper_id": fuzzy["paper_id"] if fuzzy else None,
    }


unique_input_titles = sorted({pair["paper_title"] for pair in question_pairs})
title_to_paper = {title: map_title_to_paper(title) for title in unique_input_titles}
unmatched_titles = {
    title: mapping
    for title, mapping in title_to_paper.items()
    if mapping["mapping_method"] == "unmatched"
}

{
    "input_titles": len(unique_input_titles),
    "mapped_titles": len(unique_input_titles) - len(unmatched_titles),
    "unmatched_titles": len(unmatched_titles),
    "unmatched_preview": list(unmatched_titles.items())[:5],
}


{'input_titles': 12,
 'mapped_titles': 12,
 'unmatched_titles': 0,
 'unmatched_preview': []}

## Run QA For Each Paper Title / Question Pair

Every item is appended to `results`. Unmatched titles are serialized too with `answer = None` and an explanatory `error`.

In [8]:
def answer_pair(pair: dict[str, str]) -> dict[str, Any]:
    paper_title = pair["paper_title"]
    question = pair["question"]
    mapping = title_to_paper[paper_title]

    base_result: dict[str, Any] = {
        "paper_title": paper_title,
        "matched_kg_title": mapping.get("title"),
        "paper_id": mapping.get("paper_id"),
        "paper_uri": mapping.get("paper_uri"),
        "mapping_method": mapping.get("mapping_method"),
        "mapping_score": mapping.get("mapping_score"),
        "question": question,
    }

    paper_id = mapping.get("paper_id")
    if not paper_id:
        return {
            **base_result,
            "answer": None,
            "attribute_uris": [],
            "error": "No matching KG paper id was found for this paper title.",
            "best_candidate_title": mapping.get("best_candidate_title"),
            "best_candidate_paper_id": mapping.get("best_candidate_paper_id"),
        }

    try:
        qa_result = answer_question(
            paper_id=str(paper_id),
            question=question,
            top_k=TOP_K,
            answer_chunks=ANSWER_CHUNKS,
            answer_mode=ANSWER_MODE,
        )
    except Exception as exc:
        return {
            **base_result,
            "answer": None,
            "attribute_uris": [],
            "error": f"QA failed: {type(exc).__name__}: {exc}",
        }

    return {
        **base_result,
        "answer": qa_result["answer"],
        "attribute_uris": qa_result["attribute_uris"],
        "answer_mode": qa_result.get("answer_mode", ANSWER_MODE),
        "error": None,
    }


results: list[dict[str, Any]] = []
for index, pair in enumerate(question_pairs, start=1):
    result = answer_pair(pair)
    results.append(result)
    if index % 10 == 0 or index == len(question_pairs):
        print(f"processed {index}/{len(question_pairs)}")

{
    "results": len(results),
    "answered": sum(1 for item in results if item.get("answer")),
    "unmatched_or_failed": sum(1 for item in results if item.get("error")),
    "first_result": results[0] if results else None,
}


processed 10/120
processed 20/120
processed 30/120
processed 40/120
processed 50/120
processed 60/120
processed 70/120
processed 80/120
processed 90/120
processed 100/120
processed 110/120
processed 120/120


{'results': 120,
 'answered': 120,
 'unmatched_or_failed': 0,
 'first_result': {'paper_title': 'Towards Faithfully Interpretable NLP Systems: How should we define and evaluate faithfulness?',
  'matched_kg_title': 'Towards Faithfully Interpretable NLP Systems: How should we define and evaluate faithfulness?',
  'paper_id': 'towards_faithfully_interpretable_nlp_systems_how_should_we_define_and_evaluate_faithfulness',
  'paper_uri': 'https://w3id.org/twc/sudo/kg/paper/towards_faithfully_interpretable_nlp_systems_how_should_we_define_and_evaluate_faithfulness',
  'mapping_method': 'exact',
  'mapping_score': 1.0,
  'question': 'What exactly do the authors mean by "faithfulness" in NLP interpretability?',
  'answer': 'In the context of NLP interpretability, "faithfulness" refers to the degree to which an interpretation method accurately represents the reasoning process behind a model\'s prediction. It is often treated as a binary property, where an interpretation can be classified as eithe

## Serialize Results

In [9]:
payload = {
    "source_questions_path": str(QUESTIONS_PATH),
    "output_path": str(OUTPUT_PATH),
    "total_question_pairs": len(question_pairs),
    "total_results": len(results),
    "answered": sum(1 for item in results if item.get("answer")),
    "unmatched_or_failed": sum(1 for item in results if item.get("error")),
    "title_to_paper": title_to_paper,
    "results": results,
}

OUTPUT_PATH.write_text(json.dumps(payload, indent=2, ensure_ascii=False), encoding="utf-8")

{
    "wrote": str(OUTPUT_PATH),
    "results": len(results),
    "answered": payload["answered"],
    "unmatched_or_failed": payload["unmatched_or_failed"],
}


{'wrote': '/home/desild/work/research/paper-convo-buddy/notebooks/paper_qa_batch_results.json',
 'results': 120,
 'answered': 120,
 'unmatched_or_failed': 0}